#### NourozAli
#### N10037086


In [18]:
#%pip install datasets pandas scikit-learn numpy psutil joblib


In [19]:
import os
import time
import random
import platform

import numpy as np
import pandas as pd
import psutil

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Libraries loaded successfully.")
print("Operating system:", platform.platform())
print("Python version:", platform.python_version())

Libraries loaded successfully.
Operating system: Windows-11-10.0.26300-SP0
Python version: 3.13.9


In [20]:
dataset = load_dataset("SetFit/enron_spam", split="train")

df = dataset.to_pandas()

print("Dataset loaded successfully.")
print("Number of rows:", len(df))
print("Available columns:", df.columns.tolist())

df.head()

Repo card metadata block was not found. Setting CardData to empty.


Dataset loaded successfully.
Number of rows: 31716
Available columns: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date']


,message_id,text,label,label_text,subject,message,date
0,33214,any software just for 15 $ - 99 $ understandin...,1,spam,any software just for 15 $ - 99 $,understanding oem software\nlead me not into t...,2005-06-18
1,11929,perspective on ferc regulatory action client c...,0,ham,perspective on ferc regulatory action client c...,"19 th , 2 : 00 pm edt\nperspective on ferc reg...",2001-06-19
2,19784,wanted to try ci 4 lis but thought it was way ...,1,spam,wanted to try ci 4 lis but thought it was way ...,viagra at $ 1 . 12 per dose\nready to boost yo...,2004-09-11
3,2209,"enron / hpl actuals for december 11 , 2000 tec...",0,ham,"enron / hpl actuals for december 11 , 2000",teco tap 30 . 000 / enron ; 120 . 000 / hpl ga...,2000-12-12
4,15880,looking for cheap high - quality software ? ro...,1,spam,looking for cheap high - quality software ? ro...,"water past also , burn , course . gave country...",2005-02-13


In [21]:
# Keep only the columns needed for our experiment.
df = df[["text", "label"]].copy()

# Replace missing email text with an empty string.
df["text"] = df["text"].fillna("").astype(str)

# Remove emails with no usable text.
df = df[df["text"].str.strip() != ""].copy()

# Make sure the labels are numeric.
df["label"] = pd.to_numeric(df["label"]).astype(int)

# Keep the two expected classes: 0 and 1.
df = df[df["label"].isin([0, 1])].copy()

print("Rows after basic cleaning:", len(df))
print("\nClass counts:")
print(df["label"].value_counts().sort_index())

print("\nLabel meaning:")
print("0 = Ham (legitimate email)")
print("1 = Spam")

Rows after basic cleaning: 31665

Class counts:
label
0    15553
1    16112
Name: count, dtype: int64

Label meaning:
0 = Ham (legitimate email)
1 = Spam


In [22]:

# Fixed split seed and training seeds for the group
SPLIT_SEED = 42
SEEDS = [42, 123, 2026]

# Check that enough usable emails are available
if len(df) < 5000:
    raise ValueError(
        "There are fewer than 5,000 usable emails. "
        "Check the dataset before continuing."
    )

# Step 1: Split the cleaned dataset into training,
# validation and test pools BEFORE model training.
train_pool, remaining_pool = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=SPLIT_SEED
)

validation_pool, test_pool = train_test_split(
    remaining_pool,
    test_size=0.50,
    stratify=remaining_pool["label"],
    random_state=SPLIT_SEED
)

# Step 2: Select a smaller sample from each pool.
def select_sample(data, size, seed):
    if len(data) < size:
        raise ValueError(
            f"Need {size} rows, but only have {len(data)}."
        )

    selected, _ = train_test_split(
        data,
        train_size=size,
        stratify=data["label"],
        random_state=seed
    )

    return selected.reset_index(drop=True)


train_df = select_sample(train_pool, 3500, 2026)
validation_df = select_sample(validation_pool, 750, 2026)
test_df = select_sample(test_pool, 750, 2026)

# Check the sizes
print("Training rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))

# Save the same split for all three training seeds
os.makedirs("splits", exist_ok=True)

for seed in SEEDS:
    train_df.to_csv(
        f"splits/seed_{seed}_train.csv", index=False
    )
    validation_df.to_csv(
        f"splits/seed_{seed}_validation.csv", index=False
    )
    test_df.to_csv(
        f"splits/seed_{seed}_test.csv", index=False
    )

print("Shared data splits saved successfully.")


Training rows: 3500
Validation rows: 750
Test rows: 750
Shared data splits saved successfully.


In [ ]:
"""
if len(df) < 5000:
    raise ValueError(
        "There are fewer than 5,000 usable emails. "
        "Check the dataset before continuing."
    )

sample_df, _ = train_test_split(
    df,
    train_size=5000,
    stratify=df["label"],
    random_state=2026
)

sample_df = sample_df.reset_index(drop=True)

print("Sample size:", len(sample_df))
print("\nSample class counts:")
print(sample_df["label"].value_counts().sort_index())
"""

In [ ]:
"""
os.makedirs("splits", exist_ok=True)

SEEDS = [42, 123, 2026]

for seed in SEEDS:

    # First, separate 70% training data from 30% remaining data.
    train_df, remaining_df = train_test_split(
        sample_df,
        test_size=0.30,
        stratify=sample_df["label"],
        random_state=seed
    )

    # Split the remaining 30% equally into validation and test.
    validation_df, test_df = train_test_split(
        remaining_df,
        test_size=0.50,
        stratify=remaining_df["label"],
        random_state=seed
    )

    # Save the files so all models can use identical splits.
    train_df.to_csv(
        f"splits/seed_{seed}_train.csv", index=False
    )

    validation_df.to_csv(
        f"splits/seed_{seed}_validation.csv", index=False
    )

    test_df.to_csv(
        f"splits/seed_{seed}_test.csv", index=False
    )

    print(f"\nSeed: {seed}")
    print("Training rows:", len(train_df))
    print("Validation rows:", len(validation_df))
    print("Test rows:", len(test_df))

print("\nAll split files saved in the splits folder.")
"""

In [25]:
def calculate_metrics(y_true, y_pred):
    """Calculate the main classification metrics."""

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "f1_score": f1_score(
            y_true, y_pred, zero_division=0
        )
    }


def measure_inference(predict_function, email_texts):
    """
    Measure average prediction time per email
    and the process's observed peak RAM usage.
    """

    process = psutil.Process(os.getpid())

    # Warm up the prediction function.
    predict_function([email_texts[0]])

    peak_ram = process.memory_info().rss
    total_seconds = 0.0

    # Use up to 100 emails for a repeatable timing check.
    test_emails = email_texts[:100]

    for email in test_emails:
        start = time.perf_counter()

        predict_function([email])

        total_seconds += time.perf_counter() - start

        current_ram = process.memory_info().rss
        peak_ram = max(peak_ram, current_ram)

    average_ms = (
        total_seconds / len(test_emails)
    ) * 1000

    peak_ram_mb = peak_ram / (1024 ** 2)

    return average_ms, peak_ram_mb


def count_baseline_parameters(model):
    """
    Count learned Logistic Regression coefficients
    and intercepts. This excludes TF-IDF vocabulary storage.
    """

    classifier = model.named_steps["classifier"]

    return int(
        classifier.coef_.size + classifier.intercept_.size
    )


print("Shared evaluation functions are ready.")

Shared evaluation functions are ready.


In [26]:
results = []
final_models = {}

C_VALUES = [0.1, 1.0, 10.0]

for seed in SEEDS:

    print("\n" + "=" * 50)
    print("Running baseline for seed:", seed)
    print("=" * 50)

    # Load this seed's saved splits.
    train_df = pd.read_csv(f"splits/seed_{seed}_train.csv")
    validation_df = pd.read_csv(
        f"splits/seed_{seed}_validation.csv"
    )
    test_df = pd.read_csv(f"splits/seed_{seed}_test.csv")

    X_train = train_df["text"].astype(str)
    y_train = train_df["label"].astype(int)

    X_val = validation_df["text"].astype(str)
    y_val = validation_df["label"].astype(int)

    X_test = test_df["text"].astype(str)
    y_test = test_df["label"].astype(int)

    best_c = None
    best_validation_f1 = -1

    # Tune only on training and validation data.
    for c_value in C_VALUES:

        candidate_model = Pipeline([
            (
                "tfidf",
                TfidfVectorizer(
                    max_features=10000,
                    ngram_range=(1, 2),
                    stop_words="english"
                )
            ),
            (
                "classifier",
                LogisticRegression(
                    C=c_value,
                    max_iter=1000,
                    random_state=seed
                )
            )
        ])

        candidate_model.fit(X_train, y_train)

        validation_predictions = candidate_model.predict(X_val)

        validation_f1 = f1_score(
            y_val,
            validation_predictions,
            zero_division=0
        )

        print(
            f"C={c_value} | "
            f"Validation F1={validation_f1:.4f}"
        )

        if validation_f1 > best_validation_f1:
            best_validation_f1 = validation_f1
            best_c = c_value

    print("Selected C:", best_c)
    print(
        "Best validation F1:",
        round(best_validation_f1, 4)
    )

    # Once settings are selected, refit the final model
    # using training plus validation data.
    final_training_df = pd.concat(
        [train_df, validation_df],
        ignore_index=True
    )

    final_model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                max_features=10000,
                ngram_range=(1, 2),
                stop_words="english"
            )
        ),
        (
            "classifier",
            LogisticRegression(
                C=best_c,
                max_iter=1000,
                random_state=seed
            )
        )
    ])

    final_model.fit(
        final_training_df["text"].astype(str),
        final_training_df["label"].astype(int)
    )

    # Evaluate on this seed's test set only after tuning.
    test_predictions = final_model.predict(X_test)

    metrics = calculate_metrics(y_test, test_predictions)

    # Measure inference cost.
    inference_ms, peak_ram_mb = measure_inference(
        final_model.predict,
        X_test.tolist()
    )

    # Count the classifier's learned parameters.
    parameter_count = count_baseline_parameters(final_model)

    row = {
        "model": "TF-IDF + Logistic Regression",
        "seed": seed,
        "selected_C": best_c,
        "validation_f1": best_validation_f1,
        **metrics,
        "parameter_count": parameter_count,
        "inference_ms_per_email": inference_ms,
        "observed_process_peak_ram_MB": peak_ram_mb
    }

    results.append(row)
    final_models[seed] = final_model

    print("\nFinal test results:")
    print(classification_report(
        y_test,
        test_predictions,
        labels=[0, 1],
        target_names=["Ham", "Spam"],
        zero_division=0
    ))

    print("Confusion matrix:")
    print(confusion_matrix(y_test, test_predictions))

    print("Parameter count:", parameter_count)
    print("Inference time per email (ms):", round(inference_ms, 4))
    print("Observed process peak RAM (MB):", round(peak_ram_mb, 2))

print("\nBaseline experiments completed.")


Running baseline for seed: 42
C=0.1 | Validation F1=0.9731
C=1.0 | Validation F1=0.9883
C=10.0 | Validation F1=0.9896
Selected C: 10.0
Best validation F1: 0.9896

Final test results:
              precision    recall  f1-score   support

         Ham       1.00      0.98      0.99       368
        Spam       0.98      1.00      0.99       382

    accuracy                           0.99       750
   macro avg       0.99      0.99      0.99       750
weighted avg       0.99      0.99      0.99       750

Confusion matrix:
[[361   7]
 [  1 381]]
Parameter count: 10001
Inference time per email (ms): 3.8379
Observed process peak RAM (MB): 645.2

Running baseline for seed: 123
C=0.1 | Validation F1=0.9731
C=1.0 | Validation F1=0.9883
C=10.0 | Validation F1=0.9896
Selected C: 10.0
Best validation F1: 0.9896

Final test results:
              precision    recall  f1-score   support

         Ham       1.00      0.98      0.99       368
        Spam       0.98      1.00      0.99       382



In [27]:
results_df = pd.DataFrame(results)

results_df.to_csv("baseline_results.csv", index=False)

print("Results for every seed:")
display(results_df)

print("\nResults saved to baseline_results.csv")

Results for every seed:


,model,seed,selected_C,validation_f1,accuracy,precision,recall,f1_score,parameter_count,inference_ms_per_email,observed_process_peak_ram_MB
0,TF-IDF + Logistic Regression,42,10.0,0.989556,0.989333,0.981959,0.997382,0.98961,10001,3.837947,645.195312
1,TF-IDF + Logistic Regression,123,10.0,0.989556,0.989333,0.981959,0.997382,0.98961,10001,3.434160,666.031250
2,TF-IDF + Logistic Regression,2026,10.0,0.989556,0.989333,0.981959,0.997382,0.98961,10001,1.963668,684.660156



Results saved to baseline_results.csv


In [28]:
metric_columns = [
    "accuracy",
    "precision",
    "recall",
    "f1_score",
    "inference_ms_per_email",
    "observed_process_peak_ram_MB",
    "parameter_count"
]

summary_df = results_df[metric_columns].agg(
    ["mean", "std"]
).T.reset_index()

summary_df.columns = [
    "metric",
    "mean",
    "standard_deviation"
]

summary_df.to_csv("baseline_summary.csv", index=False)

display(summary_df)

print("Summary saved to baseline_summary.csv")

PermissionError: [Errno 13] Permission denied: 'baseline_summary.csv'

In [ ]:
# Use the baseline model trained for seed 42.
model = final_models[42]

new_email = """
Congratulations! You have won a prize.
Click here now to claim your reward.
"""

prediction = model.predict([new_email])[0]

if prediction == 1:
    print("Prediction: SPAM")
else:
    print("Prediction: HAM (legitimate email)")

In [ ]:
import joblib

joblib.dump(
    final_models[42],
    "spam_baseline_model.joblib"
)

print("Saved: spam_baseline_model.joblib")

In [ ]:
loaded_model = joblib.load("spam_baseline_model.joblib")

email = ["Please review the meeting notes for tomorrow."]
prediction = loaded_model.predict(email)[0]

print("Spam" if prediction == 1 else "Ham")

In [ ]:
import sklearn
import datasets

with open("requirements.txt", "w", encoding="utf-8") as file:
    file.write(f"datasets=={datasets.__version__}\n")
    file.write(f"pandas=={pd.__version__}\n")
    file.write(f"scikit-learn=={sklearn.__version__}\n")
    file.write(f"numpy=={np.__version__}\n")
    file.write(f"psutil=={psutil.__version__}\n")
    file.write(f"joblib=={joblib.__version__}\n")

print("requirements.txt created.")

In [ ]:
import os
print(os.getcwd())
print(os.listdir())

In [ ]:

import pandas as pd
import os

print("Project folder:", os.getcwd())

results = pd.read_csv("baseline_results.csv")
summary = pd.read_csv("baseline_summary.csv")

print("\nResults for each seed:")
display(results)

print("\nSummary:")
display(summary)

print("\nSaved files:")
print(os.listdir("."))


In [ ]:
print("Column names:")
print(results.columns.tolist())

print("\nBaseline results for each seed:")
print(results[[
    "model",
    "seed",
    "selected_C",
    "validation_f1",
    "accuracy",
    "precision",
    "recall",
    "f1_score"
]].to_string(index=False))